### Регрессия для SI



### Опиание

Для регрессии SI был использован предварительно обработанный десериализированный датасет.

Дополнительно был произведен отбор целевой переменной, признаков, логарифмирование целевой переменной разделение набора данных на обучающую и тестовую выборку.

Масштабирование данных было принято решение не производить, т.к. выбранные модели в нем не нуждались: RandomForestRegressor, CatBoostRegressor не чувствительны к масштабу, LinearRegression с масштабированными признаками показала худший результат.

Как уже было сказано выше для регрессии были выбраны 3 модели: LinearRegression в качестве baseline, RandomForestRegressor, CatBoostRegressor в качестве основных моделей.

Для каждой модели был реализован подбор параметров с помощью GridSearchCV, а также кросс-валидация.

Также были рассчитаны метрики mean_absolute_error, root_mean_squared_error, r2_score.

| Модель           | MAE      | RMSE     | R²       |
|------------------|----------|----------|----------|
| LinearRegression | 0.96     | 1.24     | 0.11     |
| RandomForest     | 0.80     | 1.06     | 0.35     |
| CatBoost         | 0.82     | 1.07     | 0.33     |

Результаты означают, что LinearRegression объясняет около 11% (R²) вариации целевой переменной. Это говорит о нелинейной зависимости данных, которые линейная модель не способна уловить.

По сравнению с LinearRegression ансамблевые модели RandomForestRegressor, CatBoostRegressor показывают существенное улучшение. Модели показывают снижение MAE и RMSE, а также увеличение R² (35% у RandomForestRegressor и 33% у CatBoostRegressor), что говорит о гораздо лучшем, на 24% и 22%, объяснении моделями вариаций целевой переменной.

По сравнению с предсказанием других целевых переменных SI хуже предсказывается, потому что это не “прямая” физическая величина, а отношение двух других (CC50 / IC50) — и это усиливает выбросы и приводит к более нестабильному распределению.

In [1]:
# Устанавливаем библиотеку
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [2]:
# Импортируем необходимые библиотеки
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Загружаем DataFrame из файла
df = pd.read_pickle('my_dataset.pkl')

print(df.head())

     IC50, mM    CC50, mM         SI  MaxAbsEStateIndex  MinAbsEStateIndex  \
0    6.239374  175.482382  28.125000           5.094096           0.387225   
1    0.771831    5.402819   7.000000           3.961417           0.533868   
2  223.808778  161.142320   0.720000           2.627117           0.543231   
3    1.705624  107.855654  63.235294           5.097360           0.390603   
4  107.131532  139.270991   1.300000           5.150510           0.270476   

   MinEStateIndex       qed        SPS    MolWt  HeavyAtomMolWt  ...  \
0        0.387225  0.417362  42.928571  384.652         340.300  ...   
1        0.533868  0.462473  45.214286  388.684         340.300  ...   
2        0.543231  0.260923  42.187500  446.808         388.344  ...   
3        0.390603  0.377846  41.862069  398.679         352.311  ...   
4        0.270476  0.429038  36.514286  466.713         424.377  ...   

   fr_tetrazole  fr_thiazole  fr_thiocyan  fr_thiophene  fr_unbrch_alkane  \
0             0      

In [14]:
# Выделение признаков и целевых переменных для регрессии, логарифмирование целевых переменных для регрессии, выделение целевых переменных для классификации
y = np.log1p(df['SI'])

X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [17]:
# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Обучающая выборка: {X_train.shape[0]} строк")
print(f"Тестовая выборка: {X_test.shape[0]} строк")

Обучающая выборка: 764 строк
Тестовая выборка: 191 строк


In [23]:
# Обучаем LinearRegression с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_LR = {
    "fit_intercept": [True, False],
    "positive": [True, False]
}

grid_LR = GridSearchCV(
    LinearRegression(),
    param_grid_LR,
    cv=3,
    scoring="neg_root_mean_squared_error"
)

grid_LR.fit(X_train, y_train)

print("Best params:", grid_LR.best_params_)
print("Best CV RMSE:", -grid_LR.best_score_)

best_model_LR = grid_LR.best_estimator_
y_pred_LR = best_model_LR.predict(X_test)

mae_LR = mean_absolute_error(y_test, y_pred_LR)
rmse_LR = np.sqrt(mean_squared_error(y_test, y_pred_LR))
r2_LR = r2_score(y_test, y_pred_LR)

print("MAE:", mae_LR)
print("RMSE:", rmse_LR)
print("R2:", r2_LR)


Best params: {'fit_intercept': False, 'positive': True}
Best CV RMSE: 1.2657722345960496
MAE: 0.9606364088646377
RMSE: 1.23799304760227
R2: 0.10531960513691319


In [25]:
# Обучаем RandomForestRegressor с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_RF = {
    "n_estimators": [350, 400],
    "max_depth": [9, 10],
    "min_samples_split": [3, 4],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_RF = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid_RF,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_RF.fit(X_train, y_train)

print("Best params:", grid_RF.best_params_)
print("Best CV RMSE:", -grid_RF.best_score_)

best_model_RF = grid_RF.best_estimator_

y_pred_RF = best_model_RF.predict(X_test)

mae_RF = mean_absolute_error(y_test, y_pred_RF)
rmse_RF = np.sqrt(mean_squared_error(y_test, y_pred_RF))
r2_RF = r2_score(y_test, y_pred_RF)

print("MAE:", mae_RF)
print("RMSE:", rmse_RF)
print("R2:", r2_RF)

Best params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 400}
Best CV RMSE: 1.1496253334166875
MAE: 0.8042875738270842
RMSE: 1.057262396263528
R2: 0.3474751116256777


In [26]:
# Обучаем CatBoostRegressor с подбором параметров, кросс-валидацией и расчетом метрик
param_grid_CB = {
    "iterations": [300, 350],
    "depth": [6, 7],
    "learning_rate": [0.03, 0.05],
    "l2_leaf_reg": [3, 4]
}

grid_CB = GridSearchCV(
    CatBoostRegressor(
        random_state=42,
        verbose=0,
        loss_function="RMSE"
    ),
    param_grid_CB,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_CB.fit(X_train, y_train)

print("Best params:", grid_CB.best_params_)
print("Best CV RMSE:", -grid_CB.best_score_)

best_model_CB = grid_CB.best_estimator_

y_pred_CB = best_model_CB.predict(X_test)

mae_CB = mean_absolute_error(y_test, y_pred_CB)
rmse_CB = np.sqrt(mean_squared_error(y_test, y_pred_CB))
r2_CB = r2_score(y_test, y_pred_CB)

print("MAE:", mae_CB)
print("RMSE:", rmse_CB)
print("R2:", r2_CB)

Best params: {'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
Best CV RMSE: 1.1390556375045666
MAE: 0.8200878466512745
RMSE: 1.0741386442376128
R2: 0.3264773703214259
